In [0]:

from pyspark.sql.functions import (
    col, countDistinct, min as spark_min, max as spark_max, 
    mean, stddev, count, when, isnan, isnull, skewness, 
    percentile_approx, length, trim, upper, lower, current_date
)

# =============================================
# READ FROM UNITY CATALOG - BRONZE FINAL
# =============================================

table_name = "yelp_poc_uc.bronze_final.yelp_checkin"

# Check if the table exists and is Delta format
table_info = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
table_format = table_info["format"]
print(f"📂 Table format: {table_format}")
assert table_format.lower() == "delta", f"❌ Expected Delta table but found '{table_format}'"
print(f"✅ Confirmed: {table_name} is a Delta table")

df = spark.table(table_name)

print(f"✅ Data loaded successfully from: {table_name}")
print(f"📊 Row count: {df.count()}")
print(f"📋 Column count: {len(df.columns)}")

# =============================================
# REST OF YOUR PROFILING CODE (UNCHANGED)
# =============================================

print("="*80)
print("RAW DATA TYPES")
print("="*80)
df.printSchema()

# Collect dtype info and schema-level nullable metadata
dtypes = dict(df.dtypes)
schema_nullable = {field.name: field.nullable for field in df.schema.fields}
total_rows = df.count()

for column in df.columns:
    dtype = dtypes[column]
    n = total_rows
    unique = df.select(countDistinct(col(column))).collect()[0][0]
    null_count = df.filter(col(column).isNull()).count()
    null_pct = null_count / n * 100 if n > 0 else 0
    unique_pct = unique / n * 100 if n > 0 else 0
    col_lower = column.lower()
    
    # Sample values (up to 3)
    samples = df.select(col(column)).filter(col(column).isNotNull()).distinct().limit(3)
    sample_values = [row[0] for row in samples.collect()]
    
    # ---- CLASSIFICATION ----
    if unique == 1:
        classification = "CONSTANT"
    elif col_lower in ['id', 'host_id']:
        classification = "IDENTIFIER"
    elif dtype in ('timestamp', 'date'):
        classification = "DATETIME"
    elif col_lower in ['latitude', 'longitude']:
        classification = "COORDINATE"
    elif dtype in ('int', 'bigint'):
        min_val = df.select(spark_min(col(column))).collect()[0][0]
        if min_val is not None and min_val >= 0:
            if unique == 2:
                classification = "BINARY_FLAG"
            elif unique <= 3:
                classification = "ORDINAL"
            elif unique <= 10:
                classification = "ORDINAL"
            else:
                classification = "COUNT"
        else:
            classification = "NUMERIC_INTEGER"
    elif dtype in ('double', 'float'):
        classification = "CONTINUOUS"
    elif dtype == 'string':
        if unique <= 5:
            classification = "CATEGORICAL"
        elif unique > total_rows * 0.5:
            classification = "FREE_TEXT"
        else:
            classification = "CATEGORICAL"
    else:
        classification = "UNCLASSIFIED"
    
    print(f"\n{'='*60}")
    print(f"Column: {column}")
    print(f"{'='*60}")
    nullable = schema_nullable.get(column, None)
    print(f"  Classification:      {classification}")
    print(f"  Data Type:           {dtype}")
    print(f"  Schema Nullable:     {nullable}")
    print(f"  Unique Values:       {unique:,} ({unique_pct:.1f}%)")
    print(f"  Missing Values:      {null_count} ({null_pct:.1f}%)")
    if nullable and null_count == 0:
        print(f"  ⚠️  Schema allows NULLs but none found — consider adding a NOT NULL constraint")
    elif not nullable and null_count > 0:
        print(f"  🚨  Schema disallows NULLs but {null_count} found — data integrity issue!")
    print(f"  Sample Values:       {', '.join(str(v) for v in sample_values)}")
    
    # ---- NUMERIC PROFILING ----
    if dtype in ('int', 'bigint', 'double', 'float'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        
        mean_val = df.select(mean(col(column))).collect()[0][0]
        median_val = df.select(percentile_approx(col(column), 0.5)).collect()[0][0]
        stddev_val = df.select(stddev(col(column))).collect()[0][0]
        skew_val = df.select(skewness(col(column))).collect()[0][0]
        
        zero_count = df.filter(col(column) == 0).count()
        neg_count = df.filter(col(column) < 0).count()
        
        q1 = df.select(percentile_approx(col(column), 0.25)).collect()[0][0]
        q3 = df.select(percentile_approx(col(column), 0.75)).collect()[0][0]
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = df.filter((col(column) < lower_bound) | (col(column) > upper_bound)).count()
        
        if skew_val is not None:
            if abs(skew_val) < 0.5:
                skew_text = "Symmetric"
            elif abs(skew_val) < 1:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (Moderate)"
            else:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (High)"
        else:
            skew_text = "N/A"
        
        print(f"  Min:                 {min_val}")
        print(f"  Max:                 {max_val}")
        print(f"  Mean:                {mean_val:.2f}")
        print(f"  Median:              {median_val}")
        print(f"  Std Dev:             {stddev_val:.2f}")
        print(f"  Zero Count:          {zero_count}")
        print(f"  Negative Count:      {neg_count}")
        print(f"  Skewness:            {skew_val:.2f} ({skew_text})")
        print(f"  IQR Outliers:        {outlier_count} ({'Has outliers' if outlier_count > 0 else 'No outliers'})")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "COORDINATE":
            action = "Use for geospatial analysis"
        elif classification == "ORDINAL":
            action = "Preserve order (ordinal encoding)"
        elif classification == "BINARY_FLAG":
            action = "Use as boolean (0/1)"
        elif zero_count > total_rows * 0.9:
            action = "High zero percentage - consider log transform"
        elif skew_val and abs(skew_val) > 1:
            action = "Scale & check outliers (high skew)"
        else:
            action = "Scale & check outliers"
        print(f"  Action:              {action}")
    
    # ---- STRING PROFILING ----
    elif dtype == 'string':
        empty_count = df.filter((col(column) == "") | (col(column).isNull())).count()
        whitespace_count = df.filter(length(col(column)) != length(trim(col(column)))).count()
        
        total_non_null = n - null_count
        if total_non_null > 0:
            all_upper = df.filter(col(column) == upper(col(column))).count() - null_count
            all_lower = df.filter(col(column) == lower(col(column))).count() - null_count
            upper_pct = all_upper / total_non_null * 100
            lower_pct = all_lower / total_non_null * 100
            
            if upper_pct > 80:
                casing = f"{upper_pct:.0f}% Uppercase"
            elif lower_pct > 80:
                casing = f"{lower_pct:.0f}% Lowercase"
            else:
                casing = "Mixed Casing"
        else:
            casing = "N/A (all null)"
        
        len_stats = df.select(
            spark_min(length(col(column))).alias("min_len"),
            spark_max(length(col(column))).alias("max_len"),
            mean(length(col(column))).alias("avg_len")
        ).collect()[0]
        
        print(f"  Empty Count:         {empty_count}")
        print(f"  Whitespace Issues:   {whitespace_count}")
        print(f"  Casing:              {casing}")
        print(f"  Length:              Min: {len_stats['min_len']}, Max: {len_stats['max_len']}, Avg: {len_stats['avg_len']:.2f}")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "FREE_TEXT":
            action = "Use for text mining / NLP only"
        elif unique <= 5:
            action = "One-hot or frequency encode"
        elif unique <= 50:
            action = "Label encode or one-hot (careful with cardinality)"
        else:
            action = "Frequency encode or target encode"
        print(f"  Action:              {action}")
    
    # ---- DATETIME PROFILING ----
    elif dtype in ('timestamp', 'date'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        future_count = df.filter(col(column) > current_date()).count()
        old_count = df.filter(col(column) < "2008-01-01").count()
        
        print(f"  Date Range:          {min_val} to {max_val}")
        print(f"  Future Dates:        {future_count}")
        print(f"  Before 2008:         {old_count}")
        print(f"  Action:              Convert to datetime, extract features")

print(f"\n{'='*80}")
print("COMPREHENSIVE DATA PROFILING COMPLETE")
print(f"{'='*80}")

📂 Table format: delta
✅ Confirmed: yelp_poc_uc.bronze_final.yelp_checkin is a Delta table
✅ Data loaded successfully from: yelp_poc_uc.bronze_final.yelp_checkin
📊 Row count: 131930
📋 Column count: 2
RAW DATA TYPES
root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)


Column: business_id
  Classification:      FREE_TEXT
  Data Type:           string
  Unique Values:       131,930 (100.0%)
  Missing Values:      0 (0.0%)
  Sample Values:       -1ygry5D2oOt_vch4rjMuA, -2aGyAUenQEZWAtNUnMHzg, -2j8XqQL68DPqYiafH9JXg
  Empty Count:         0
  Whitespace Issues:   0
  Casing:              Mixed Casing
  Length:              Min: 22, Max: 22, Avg: 22.00
  Action:              Use for text mining / NLP only

Column: date
  Classification:      FREE_TEXT
  Data Type:           string
  Unique Values:       131,930 (100.0%)
  Missing Values:      0 (0.0%)
  Sample Values:       2010-06-28 22:25:18, 2010-07-21 23:03:49, 2010-10-25 21:16:49, 2010-11-11 22:18:02, 201